# Week 1 Lab — Arrays and tidy data
**Module 1 · Python & data toolkit · Ladder L1 (Tool fluent)**

| Session | Part of this notebook | Time |
|---|---|---|
| Thu (practise) | Parts A–B: shapes, broadcasting, vectorized z-score | 1.5 h |
| Sat (build) | Parts C–F: download TCGA-LUAD from UCSC Xena, load, join expression + clinical, save a tidy table | 3–4 h |

**You will be able to:** predict array shapes before running code; vectorize a per-gene z-score; load a Xena expression matrix and clinical table; join them on a validated key; explain why one patient can have several samples.

### Split strategy (stated before any training)
No model is trained this week. But two habits start now:
1. **Statistics computed here (gene means / SDs) are for exploration only.** When we model (Module 3+), scaling parameters must be fitted on the **training split only**, inside a `Pipeline`.
2. **The future split unit is the patient, not the sample.** TCGA patients can have a tumour *and* a normal sample. Part E counts how many — this is the leakage risk we will guard against with `GroupKFold` in Module 4.

### Leakage check for this week
- [ ] No scaling/normalisation statistics are saved and reused for modelling.
- [ ] The tidy table keeps a `patient_id` column so later splits can be grouped.

## 0. Setup — pinned versions

In [ ]:
# If running outside the conda env:  pip install numpy==2.3.4 pandas==2.3.3 matplotlib==3.10.7 seaborn==0.13.2 scikit-learn==1.7.2 pyarrow==21.0.0
import sys, time, gzip, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import StandardScaler

PINNED = {"numpy": "2.3.4", "pandas": "2.3.3", "matplotlib": "3.10.7", "seaborn": "0.13.2", "scikit-learn": "1.7.2"}
found = {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
         "seaborn": sns.__version__, "scikit-learn": sklearn.__version__}
print("python", sys.version.split()[0])
for k, v in PINNED.items():
    flag = "OK " if found[k] == v else "DIFF"
    print(f"{flag} {k:13s} pinned={v:8s} found={found[k]}")

RNG = np.random.default_rng(42)
DATA = Path("data"); DATA.mkdir(exist_ok=True)

## Part A — Shapes and broadcasting (Thu, ~30 min)

Convention for this course: **expression matrices are `genes × samples` on disk** (Xena, most GEO tables) and **`samples × genes` for ML** (scikit-learn expects rows = observations). Always print `.shape` after loading and after transposing.

Broadcasting rule: compare shapes from the **right**. Two dimensions are compatible if they are equal or one of them is 1. The size-1 dimension is stretched (virtually, no copy).

In [ ]:
# Toy matrix: 3 genes x 4 samples
X = np.array([[2., 4., 6., 8.],
              [1., 1., 4., 2.],
              [10., 12., 11., 7.]])
print("X shape:", X.shape)

gene_means_1d = X.mean(axis=1)                 # shape (3,)
gene_means_2d = X.mean(axis=1, keepdims=True)  # shape (3, 1)
print("1-D means:", gene_means_1d.shape, " 2-D means:", gene_means_2d.shape)

# Works: (3,4) - (3,1) -> (3,4): each gene's mean is subtracted across its row
print((X - gene_means_2d).round(2))

In [ ]:
# The classic bug: (3,4) - (3,) aligns the 3 with the LAST axis (4) -> error.
try:
    X - gene_means_1d
except ValueError as e:
    print("ValueError:", e)

# Worse: a SQUARE matrix silently gives the wrong answer
S = np.arange(9.).reshape(3, 3)
wrong = S - S.mean(axis=1)                  # subtracts column-wise!
right = S - S.mean(axis=1, keepdims=True)
print("wrong row means:", wrong.mean(axis=1), " right row means:", right.mean(axis=1))

**TODO A1.** Before running, write the shape of each result as a comment, then run to check.

In [ ]:
a = np.ones((5, 1)); b = np.ones((1, 7)); c = np.ones((7,)); d = np.ones((5, 7))
# TODO A1: predict, then uncomment
# print((a + b).shape)   # predict: ?
# print((d + c).shape)   # predict: ?
# print((d * a).shape)   # predict: ?
# print((d + b.T).shape) # predict: ?  (error or shape?)

## Part B — Vectorized z-score over genes × samples (Thu, ~60 min)

Per-gene z-score: $z_{gs} = \dfrac{x_{gs} - \mu_g}{\sigma_g}$, with $\mu_g = \frac{1}{S}\sum_s x_{gs}$ and $\sigma_g = \sqrt{\frac{1}{S - \delta}\sum_s (x_{gs}-\mu_g)^2}$, where $\delta$ is `ddof` (0 = population, NumPy's default and what `StandardScaler` uses; 1 = sample SD, pandas' default).

In [ ]:
def zscore_loop(X):
    # X: genes x samples. Plain-Python reference implementation.
    G, S = X.shape
    Z = np.empty_like(X, dtype=float)
    for g in range(G):
        mu = sum(X[g, s] for s in range(S)) / S
        var = sum((X[g, s] - mu) ** 2 for s in range(S)) / S
        sd = var ** 0.5
        for s in range(S):
            Z[g, s] = (X[g, s] - mu) / sd if sd > 0 else 0.0
    return Z

def zscore_vec(X, ddof=0):
    # X: genes x samples. Vectorized; zero-variance genes -> 0.
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, ddof=ddof, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        Z = (X - mu) / sd
    return np.where(sd > 0, Z, 0.0)

# Hand-checkable example (matches the worked example in the study guide)
toy = np.array([[2., 4., 6.], [1., 1., 4.]])
print(zscore_vec(toy).round(3))

In [ ]:
# Simulated log-expression: 2,000 genes x 300 samples
Xsim = RNG.normal(loc=RNG.uniform(0, 12, size=(2000, 1)), scale=RNG.uniform(0.2, 3, size=(2000, 1)), size=(2000, 300))
Xsim[0] = 5.0  # a zero-variance gene, as happens with filtered-out genes

t0 = time.perf_counter(); Zl = zscore_loop(Xsim); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); Zv = zscore_vec(Xsim); t_vec = time.perf_counter() - t0
print(f"loop {t_loop:.2f}s  vectorized {t_vec*1000:.1f} ms  speed-up x{t_loop/t_vec:.0f}")
print("max abs diff:", np.abs(Zl - Zv).max())

In [ ]:
# Check against scikit-learn. StandardScaler standardizes COLUMNS (features),
# so it needs samples x genes -> transpose in, transpose out.
Zsk = StandardScaler().fit_transform(Xsim.T).T
print("matches sklearn:", np.allclose(Zv, Zsk))
# pandas .std() defaults to ddof=1 -> slightly different values
Zpd = pd.DataFrame(Xsim).pipe(lambda d: d.sub(d.mean(axis=1), axis=0).div(d.std(axis=1), axis=0)).to_numpy()
print("pandas ddof=1 vs numpy ddof=0, max diff:", np.nanmax(np.abs(Zpd - Zv)).round(4))

**TODO B1.** Write `zscore_samples(X)` that z-scores each **sample** (column) instead of each gene. Verify every column has mean ≈ 0 and SD ≈ 1.

**TODO B2.** Write `robust_z(X)` using the median and MAD ($\mathrm{MAD}_g = \mathrm{median}_s |x_{gs} - \mathrm{median}_g|$, scaled by 1.4826). Only broadcasting — no loops.

**TODO B3.** Memory: how many bytes does `Xsim` use? How much would a 60,000 × 10,000 float64 matrix need? What does `astype(np.float32)` save?

In [ ]:
# TODO B1

In [ ]:
# TODO B2

In [ ]:
# TODO B3

## Part C — Download a TCGA cohort from UCSC Xena (Sat)

Cohort: **TCGA Lung Adenocarcinoma (LUAD)**, legacy "sampleMap" datasets on the TCGA hub.
- Expression: `TCGA.LUAD.sampleMap/HiSeqV2` — gene-level RSEM, **already log2(norm_count + 1)**, genes × samples.
- Clinical: `TCGA.LUAD.sampleMap/LUAD_clinicalMatrix` — samples × variables.

**Exact steps (manual route, recommended first time):**
1. Open https://xenabrowser.net/datapages/ → hub **TCGA Hub** → cohort **TCGA Lung Adenocarcinoma (LUAD)**.
2. Click the *gene expression RNAseq → IlluminaHiSeq* dataset (`HiSeqV2`). Copy the **download** link; save the file as `data/HiSeqV2.gz`.
3. Back in the cohort, open the *phenotype → Phenotypes* dataset (`LUAD_clinicalMatrix`); save as `data/LUAD_clinicalMatrix.tsv`.
4. Record the download date and the link in `data/README.md` (provenance).

The cell below tries the direct links (the pattern Xena uses for this hub — **check them against the download links you copied**). If downloading fails (offline, firewall), set `USE_SYNTHETIC = True` to run the rest of the lab on a simulated cohort with the same structure.

In [ ]:
USE_SYNTHETIC = False   # set True if you cannot reach Xena

HUB = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/"
FILES = {
    "expr": ("TCGA.LUAD.sampleMap%2FHiSeqV2.gz", DATA / "HiSeqV2.gz"),
    "clin": ("TCGA.LUAD.sampleMap%2FLUAD_clinicalMatrix", DATA / "LUAD_clinicalMatrix.tsv"),
}

def fetch(remote, local):
    if local.exists():
        print("cached:", local); return True
    try:
        print("downloading", remote, "...")
        urllib.request.urlretrieve(HUB + remote, local)
        return True
    except Exception as e:
        print("FAILED:", e); return False

if not USE_SYNTHETIC:
    ok = all(fetch(r, l) for r, l in FILES.values())
    if not ok:
        print("-> falling back to synthetic data"); USE_SYNTHETIC = True

In [ ]:
def make_synthetic_cohort(n_patients=200, n_genes=3000, seed=0):
    # Mimics Xena layout: expression genes x samples (log2 scale), clinical samples x vars.
    rng = np.random.default_rng(seed)
    patients = [f"TCGA-{rng.integers(10,99)}-{i:04d}" for i in range(n_patients)]
    samples = [p + "-01" for p in patients]                                   # primary tumour
    samples += [p + "-11" for p in rng.choice(patients, 30, replace=False)]   # some matched normals
    genes = [f"GENE{i}" for i in range(n_genes)]
    base = rng.uniform(0, 12, size=(n_genes, 1))
    expr = pd.DataFrame(rng.normal(base, 1.0, size=(n_genes, len(samples))).clip(0), index=genes, columns=samples)
    expr.index.name = "sample"   # Xena puts 'sample' in the corner cell
    clin_samples = samples[:-5] + [p + "-01" for p in [f"TCGA-00-{9000+i}" for i in range(4)]]  # 5 missing, 4 extra
    clin = pd.DataFrame({
        "sampleID": clin_samples,
        "sample_type": ["Primary Tumor" if s.endswith("-01") else "Solid Tissue Normal" for s in clin_samples],
        "gender": rng.choice(["MALE", "FEMALE"], len(clin_samples)),
        "age_at_initial_pathologic_diagnosis": rng.integers(38, 88, len(clin_samples)).astype(float),
        "pathologic_stage": rng.choice(["Stage IA", "Stage IB", "Stage IIA", "Stage IIB", "Stage IIIA", "Stage IV", np.nan], len(clin_samples)),
    })
    clin.loc[rng.choice(len(clin), 8, replace=False), "age_at_initial_pathologic_diagnosis"] = np.nan
    return expr, clin

if USE_SYNTHETIC:
    expr_raw, clin_raw = make_synthetic_cohort()
    print("SYNTHETIC cohort generated")

## Part D — Load and orient (Sat)
Xena expression files: first column = gene symbol, header row = sample IDs, tab-separated, gzip-compressed. Clinical matrix: one row per **sample** (`sampleID`), many columns, lots of missing values.

In [ ]:
if not USE_SYNTHETIC:
    expr_raw = pd.read_csv(FILES["expr"][1], sep="\t", index_col=0, compression="gzip")
    clin_raw = pd.read_csv(FILES["clin"][1], sep="\t", low_memory=False)

print("expression (genes x samples):", expr_raw.shape)
print("clinical   (samples x vars) :", clin_raw.shape)
print("value range:", float(np.nanmin(expr_raw.values)), "to", float(np.nanmax(expr_raw.values)), "(log2 scale expected: ~0-20)")
expr_raw.iloc[:3, :4]

In [ ]:
# Sanity checks you should ALWAYS run on a fresh matrix
assert expr_raw.index.is_unique, "duplicate gene symbols"
assert expr_raw.columns.is_unique, "duplicate sample IDs"
print("NaNs in expression:", int(expr_raw.isna().sum().sum()))
print("zero-variance genes:", int((expr_raw.std(axis=1) == 0).sum()))

# Orient for ML: samples x genes
X = expr_raw.T
X.index.name = "sample_id"
print("X (samples x genes):", X.shape)

## Part E — Parse TCGA barcodes and join with clinical (Sat)
A Xena TCGA sample ID looks like `TCGA-05-4244-01`:
- characters 1–12 → **patient** (`TCGA-05-4244`)
- characters 14–15 → **sample type code**: `01` primary tumour, `02` recurrent tumour, `06` metastatic, `11` solid tissue normal.

In [ ]:
def parse_barcode(ids):
    ids = pd.Series(ids, name="sample_id")
    return pd.DataFrame({
        "sample_id": ids.values,
        "patient_id": ids.str[:12].values,
        "type_code": ids.str[13:15].values,
    })

meta = parse_barcode(X.index)
code_map = {"01": "tumor", "02": "tumor_recurrent", "06": "metastatic", "11": "normal"}
meta["tissue"] = meta["type_code"].map(code_map).fillna("other")
print(meta["tissue"].value_counts())

per_patient = meta.groupby("patient_id").size()
print(f"\npatients: {per_patient.size}, samples: {len(meta)}, patients with >1 sample: {(per_patient > 1).sum()}")

In [ ]:
# Clean the clinical table: keep a few columns, rename, check the key
keep = {"sampleID": "sample_id", "sample_type": "sample_type", "gender": "sex",
        "age_at_initial_pathologic_diagnosis": "age", "pathologic_stage": "stage"}
missing_cols = [c for c in keep if c not in clin_raw.columns]
print("missing clinical columns:", missing_cols or "none")
clin = clin_raw[[c for c in keep if c in clin_raw.columns]].rename(columns=keep)
assert clin["sample_id"].is_unique, "clinical has duplicate sample IDs"

# Which IDs are in one table but not the other? (always look before joining)
in_expr, in_clin = set(meta.sample_id), set(clin.sample_id)
print("expr only:", len(in_expr - in_clin), " clin only:", len(in_clin - in_expr), " both:", len(in_expr & in_clin))

In [ ]:
# Join: left join from the expression side; validate cardinality
tidy_meta = meta.merge(clin, on="sample_id", how="left", validate="one_to_one", indicator=True)
print(tidy_meta["_merge"].value_counts())
tidy_meta = tidy_meta.drop(columns="_merge")

# Consistency check: barcode-derived tissue vs clinical sample_type
print(pd.crosstab(tidy_meta["tissue"], tidy_meta["sample_type"].fillna("NA")))

**TODO E1.** Run the same merge with `how="inner"` and `how="outer"`. Report the row counts and explain each difference.

**TODO E2.** Build `patient_table`: one row per patient with columns `n_samples`, `has_normal`, `sex`, `age`. (Hint: `groupby(...).agg(...)`.)

**TODO E3.** Convert `stage` into an ordered categorical with collapsed levels I/II/III/IV (strip sub-letters A/B). How many patients have missing stage?

In [ ]:
# TODO E1

In [ ]:
# TODO E2

In [ ]:
# TODO E3

## Part F — First look and save (Sat)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
tidy_meta["tissue"].value_counts().plot.bar(ax=axes[0], title="Samples by tissue")
X.mean(axis=1).plot.hist(bins=40, ax=axes[1], title="Per-sample mean log2 expression")
tidy_meta["age"].plot.hist(bins=20, ax=axes[2], title="Age at diagnosis")
plt.tight_layout(); plt.show()

In [ ]:
# Z-score the real matrix with YOUR function (exploration only — see leakage note at top)
Z = zscore_vec(X.T.to_numpy()).T          # back to samples x genes
print("Z shape:", Z.shape, " per-gene mean ~0:", np.allclose(Z.mean(axis=0), 0, atol=1e-8))

# Save tidy outputs (expression stays samples x genes; metadata separate, joined by sample_id)
X.astype("float32").to_parquet(DATA / "luad_expr_samples_x_genes.parquet")
tidy_meta.to_csv(DATA / "luad_sample_metadata.csv", index=False)
print("saved:", sorted(p.name for p in DATA.iterdir()))

**TODO F1.** Write `data/README.md`: source URLs, download date, dataset versions shown on the Xena page, units (log2(norm_count+1)), and the shape of each saved file.

**TODO F2 (stretch).** Rewrite one loop-heavy function from your own WGS/ETL code with vectorization; time both versions and paste the numbers into your learning log.

---
# Solutions
Try the TODOs first. Solutions run on whichever data (real or synthetic) is loaded.

In [ ]:
# A1
a = np.ones((5, 1)); b = np.ones((1, 7)); c = np.ones((7,)); d = np.ones((5, 7))
print((a + b).shape)   # (5, 7)  both stretched
print((d + c).shape)   # (5, 7)  c treated as (1, 7)
print((d * a).shape)   # (5, 7)
try:
    print((d + b.T).shape)
except ValueError as e:
    print("error:", e)  # (5,7) vs (7,1): 7 vs 1 ok, 5 vs 7 fail

In [ ]:
# B1 — per-sample z-score
def zscore_samples(X):
    mu = X.mean(axis=0, keepdims=True); sd = X.std(axis=0, keepdims=True)
    return (X - mu) / sd
Zs = zscore_samples(Xsim)
print(np.allclose(Zs.mean(axis=0), 0), np.allclose(Zs.std(axis=0), 1))

# B2 — robust z-score (median / MAD)
def robust_z(X, eps=1e-12):
    med = np.median(X, axis=1, keepdims=True)
    mad = 1.4826 * np.median(np.abs(X - med), axis=1, keepdims=True)
    return np.where(mad > eps, (X - med) / np.maximum(mad, eps), 0.0)
print(robust_z(Xsim).shape)

# B3 — memory
print(Xsim.nbytes / 1e6, "MB")
print(60_000 * 10_000 * 8 / 1e9, "GB as float64;", 60_000 * 10_000 * 4 / 1e9, "GB as float32")

In [ ]:
# E1
for how in ["left", "inner", "outer"]:
    print(how, len(meta.merge(clin, on="sample_id", how=how)))
# left = all expression samples; inner drops expression samples lacking clinical rows;
# outer adds clinical-only samples (no expression) -> rows with all-NaN expression if joined to X.

# E2
patient_table = tidy_meta.groupby("patient_id").agg(
    n_samples=("sample_id", "size"),
    has_normal=("tissue", lambda s: (s == "normal").any()),
    sex=("sex", "first"),
    age=("age", "first"),
)
print(patient_table.head()); print(patient_table["has_normal"].value_counts())

# E3
stage = (tidy_meta["stage"].astype("string")
         .str.extract(r"Stage\s+(IV|III|II|I)", expand=False))
tidy_meta["stage_simple"] = pd.Categorical(stage, categories=["I", "II", "III", "IV"], ordered=True)
print(tidy_meta["stage_simple"].value_counts(dropna=False))
print("patients with missing stage:", tidy_meta.groupby("patient_id")["stage_simple"].first().isna().sum())